In [ ]:
!pip install -U langchain langchain-openai langgraph

In [ ]:
#generate key from here https://platform.openai.com/api-keys
import os
os.environ["OPENAI_API_KEY"] = " "

key = os.environ.get("OPENAI_API_KEY")
print("Key starts with:", key[:12] if key else "NOT SET")
print("Key length:", len(key) if key else 0)

In [ ]:
#1. LangChain — LLM + one tool
#openai.AuthenticationError: Incorrect API key provided
#export OPENAI_API_KEY="sk-..."      # Mac/Linux
#setx OPENAI_API_KEY "sk-..."        # Windows (restart terminal after)
#!pip install -U langchain langchain-openai langgraph

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

llm = ChatOpenAI(model="gpt-4o-mini")
agent = create_agent(llm, [add])

result = agent.invoke({"messages": [("user", "What is 5 + 7?")]})
print(result["messages"][-1].content)

In [ ]:
#billing issue 
from openai import OpenAI
client = OpenAI()

try:
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "hi"}]
    )
    print(resp.choices[0].message.content)
except Exception as e:
    print(type(e).__name__, ":", e)

In [ ]:
#2. AutoGen — two agents talking to each other
!pip install -U autogen-agentchat "autogen-ext[openai]"

In [ ]:
# Conversation between agents
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

model = OpenAIChatCompletionClient(model="gpt-4o-mini")
poet = AssistantAgent("poet", model, system_message="Write a 2-line poem.")
critic = AssistantAgent("critic", model, system_message="Say APPROVE if good.")

team = RoundRobinGroupChat([poet, critic], termination_condition=TextMentionTermination("APPROVE"))

In [ ]:
#to see agents conversation,facing error due to insufficient_quota billing

import asyncio
from autogen_agentchat.ui import Console

async def main():
    await Console(team.run_stream(task="Write a poem about the ocean."))

await main()   # use this in Jupyter (NOT asyncio.run(main()))

In [ ]:
##3. CrewAI — agents with roles doing tasks
!python -m pip install --upgrade --ignore-installed pywin32

In [ ]:
pip install -U crewai

In [ ]:
#3. CrewAI — agents with roles doing tasks
from crewai import Agent, Task, Crew

researcher = Agent(role="Researcher", goal="Find facts", backstory="You love data.")
writer = Agent(role="Writer", goal="Summarize clearly", backstory="You write for beginners.")

In [ ]:
#4. BabyAGI — a task queue that grows itself
from collections import deque

tasks = deque(["Find 3 facts about the moon"])

while tasks:
    task = tasks.popleft()
    result = f"Fake result for: {task}"          # normally: call the LLM here
    print(result)

    # LLM would normally generate this list based on the result
    new_tasks = [f"Follow-up on: {task}"]
    tasks.extend(new_tasks)

    if len(tasks) > 3:      # stop condition for the demo
        break

In [ ]:
#5. AutoGPT — a goal-driven loop that decides when it's done
goal = "Find the population of France"
steps = 0

while steps < 5:
    steps += 1
    # normally: ask the LLM "what should I do next, or am I done?"
    thought = "I should search for the population."
    action = "search('population of France')"
    observation = "68 million"                 # result of running the action

    print(thought, "->", action, "->", observation)

    if "million" in observation:                # LLM would decide this itself
        print("Goal achieved:", observation)
        break

In [ ]:
"""
Example 1: Simple Reflex Agent (Agent AI)

A basic thermostat agent that reacts to the current temperature reading.
This shows the SIMPLEST form of an "Agent AI" - it has no memory, no
planning, and no reasoning. It just maps a percept directly to an action
using fixed rules (condition-action rules).
"""


class SimpleReflexAgent:
    def __init__(self, target_temp=22):
        self.target_temp = target_temp

    def act(self, current_temp):
        """Decide an action based only on the current percept."""
        if current_temp < self.target_temp - 1:
            return "Turn ON heater"
        elif current_temp > self.target_temp + 1:
            return "Turn ON cooler"
        else:
            return "Do nothing (temperature OK)"


if __name__ == "__main__":
    agent = SimpleReflexAgent(target_temp=22)

    # Simulate a stream of sensor readings
    sensor_readings = [18, 19, 21, 22, 23, 26, 28, 24, 22, 20]

    print("Simple Reflex Agent - Thermostat Simulation")
    print("Target Temperature: 22 C")
    print("-" * 45)

    for step, temp in enumerate(sensor_readings, start=1):
        action = agent.act(temp)
        print(f"Step {step:2d} | Sensor Temp: {temp:3d} C | Action: {action}")

In [ ]:
"""
Example 2: Goal-Based Agent with Memory (Agent AI)

A vacuum-cleaning agent that keeps an internal state (memory of which
rooms are clean) and chooses actions to satisfy a GOAL: "all rooms clean".

This is more advanced than a reflex agent because it uses memory and a
simple decision process, but it still does NOT plan multiple steps ahead
or use an LLM to reason - it is still classic "Agent AI".
"""


class GoalBasedVacuumAgent:
    def __init__(self, rooms):
        # Internal memory / model of the world
        self.room_status = {room: "Dirty" for room in rooms}
        self.location = rooms[0]
        self.rooms = rooms

    def perceive_and_act(self):
        actions = []
        for room in self.rooms:
            self.location = room
            if self.room_status[room] == "Dirty":
                self.room_status[room] = "Clean"
                actions.append(f"Move to {room} -> Clean it")
            else:
                actions.append(f"Move to {room} -> Already clean, skip")
        return actions

    def goal_satisfied(self):
        return all(status == "Clean" for status in self.room_status.values())


if __name__ == "__main__":
    rooms = ["Room-A", "Room-B", "Room-C"]
    agent = GoalBasedVacuumAgent(rooms)

    print("Goal-Based Agent - Vacuum Cleaning Simulation")
    print(f"Goal: All rooms clean {rooms}")
    print("-" * 50)

    actions = agent.perceive_and_act()
    for a in actions:
        print(a)

    print("-" * 50)
    print("Final Room Status:", agent.room_status)
    print("Goal Achieved:", agent.goal_satisfied())

In [ ]:
#Example 3: Agentic AI - Reasoning + Tool-Use
Loop (ReAct Pattern)

In [ ]:
import re

# ---------- TOOLS the agent is allowed to use ----------

def calculator_tool(expression):
    try:
        # NOTE: eval is used here only for a controlled classroom demo
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


def knowledge_base_tool(query):
    fake_kb = {
        "capital of india": "New Delhi",
        "speed of light": "299792458 m/s",
        "population of india 2024": "approximately 1.44 billion",
    }
    return fake_kb.get(query.lower().strip(), "No data found in knowledge base")


TOOLS = {
    "calculator": calculator_tool,
    "knowledge_base": knowledge_base_tool,
}


# ---------- The "Brain" (stands in for an LLM's reasoning) ----------

def agent_brain(goal, step, scratchpad):
    """
    Decides the next Thought + Action given the goal and history so far.
    A real agentic AI system replaces this function with a call to an
    LLM such as Claude, which generates the Thought/Action text itself.
    """
    if step == 1:
        thought = "I need to find the population of India, then divide it by 1000."
        action = ("knowledge_base", "population of india 2024")
    elif step == 2:
        thought = "I have the population text. Now I should extract the number and divide by 1000 using the calculator."
        action = ("calculator", "1440000000 / 1000")
    else:
        thought = "I now have the final numeric answer, task is complete."
        action = ("finish", None)

    return thought, action


def run_agent(goal, max_steps=5):
    print(f"GOAL: {goal}\n")
    scratchpad = []

    for step in range(1, max_steps + 1):
        thought, (tool_name, tool_input) = agent_brain(goal, step, scratchpad)

        print(f"Step {step}")
        print(f"  Thought : {thought}")

        if tool_name == "finish":
            print(f"  Action  : Finish")
            print("\nFinal Answer: India's population divided by 1000 is "
                  f"{scratchpad[-1]['observation']}")
            break

        observation = TOOLS[tool_name](tool_input)
        print(f"  Action  : Use tool '{tool_name}' with input '{tool_input}'")
        print(f"  Observation: {observation}\n")

        scratchpad.append({
            "tool": tool_name,
            "input": tool_input,
            "observation": observation,
        })


if __name__ == "__main__":
    run_agent("Find India's 2024 population and divide it by 1000.")